# Survey Analysis Pipeline
## End-to-End Sensory Research Reporting Toolkit

In [ ]:
"""
=============================================================
 SURVEY ANALYSIS PIPELINE
 End-to-End Sensory Research Reporting Toolkit
=============================================================
A universal Python pipeline for processing and analysing
consumer sensory survey data across multiple products,
visits, and markets.

What this pipeline does:
  Ingests raw survey CSV exports, cleans and maps Likert
  responses, runs a full battery of statistical tests, and
  generates formatted Excel reports with charts.

  Reduced report turnaround from 1.5 days to 3 hours.
  Eliminated dependency on SPSS entirely.

Pipeline sections:
  1 — Configuration & Setup
  2 — Data Ingestion & Cleaning
  3 — Data Reshaping & Summarisation
  4 — Statistical Testing
  5 — Sensory Analysis (JAR, CATA, MaxDiff)
  6 — Driver Analysis
  7 — Translation & NLP

Inputs:
  - Raw survey CSV exports (one file per visit/product)
  - JSON config file (column lists, labels, abbreviations)
  - JSON Likert level mappings (likert_levels.json)
  - HDF5 intermediate dataframe store (dataframes.h5)

Outputs:
  - Excel reports per product and visit
  - Statistical test results (Mann-Whitney U, McNemar,
    Wilcoxon, Fisher's Exact)
  - Driver analysis (LR, Johnson, RF, HGB, Lasso)
  - JAR and CATA penalty analysis with charts
  - Translated open-ended responses

Configuration:
  Set paths and API key in SECTION 1 before running.
  All study-specific column lists are loaded from a JSON
  config file — see the configuration cell for details.
=============================================================
"""

---
## SECTION 1 — Configuration & Setup
Set paths, API keys, and load study configuration before running any other section.

In [ ]:
import os
import re
import json
import csv
import sys
import logging
import textwrap
import warnings
import sqlite3
import math
import itertools
from contextlib import contextmanager
from collections import defaultdict, Counter
from typing import List, Optional, Dict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
from pandas.plotting import scatter_matrix

import scipy.stats as stats
from scipy.stats import chi2_contingency, chi2, fisher_exact, ttest_ind
from scipy.stats import mannwhitneyu
import statsmodels.api as sm
from statsmodels.stats.contingency_tables import mcnemar

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment

import chardet
import requests
from deep_translator import GoogleTranslator
from langdetect import detect
from textblob import TextBlob
from fuzzywuzzy import process

warnings.filterwarnings('ignore')
rcParams['figure.figsize'] = 15, 8
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

print('Imports complete')

In [ ]:
# ============================================================
# PATH CONFIGURATION
# Set these before running the pipeline
# ============================================================

# Root path for this study — all outputs save here
path = ""  # e.g. 'C:/projects/study_name/'

# HDF5 intermediate dataframe store
# Created automatically on first use if it does not exist
h5_file_path = "data/dataframes.h5"  # update to your local path

# Google Translate API key
# Set via environment variable — never hardcode in source
GOOGLE_TRANSLATE_API_KEY = os.environ.get("GOOGLE_TRANSLATE_API_KEY", "")

print(f"path         : {path}")
print(f"h5_file_path : {h5_file_path}")
print(f"API key set  : {'Yes' if GOOGLE_TRANSLATE_API_KEY else 'No — set GOOGLE_TRANSLATE_API_KEY env var'}")

In [ ]:
# ============================================================
# STUDY CONFIGURATION
# Load column lists and labels from the study JSON config file.
# The JSON file is study-specific and defines which columns
# belong to which analysis group.
#
# Required keys in the JSON:
#   TEN     — 10-point scale columns
#   T2B     — top-2-box columns (5-point)
#   JAR     — JAR scale columns
#   T2B_JAR — combined 5-point T2B and JAR columns
#   CATA    — Check-All-That-Apply columns
#   MD      — MaxDiff columns
#   SKIP    — columns to exclude from analysis
#   OE      — open-ended text columns
#   T2B_TEN — combined T2B and 10-point columns for correlation
#   column_pairs_dict — crosstab column pairs
# ============================================================

config_file = f"{path}study_config.json"  # update filename to match your study

with open(config_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

all_10pt_cols     = data['TEN']
T2B_cols          = data['T2B']
jar_cols          = data['JAR']
all_5pt_cols      = data['T2B_JAR']
CATA_cols         = data['CATA']
md_cols           = data.get('MD', [])
columns_to_skip   = data['SKIP']
OE                = data['OE']
T2B_TEN           = data['T2B_TEN']
column_pairs_dict = data['column_pairs_dict']
columns_to_drop   = data.get('drop_cols', [])

print(f"10pt columns : {len(all_10pt_cols)}")
print(f"T2B columns  : {len(T2B_cols)}")
print(f"JAR columns  : {len(jar_cols)}")
print(f"CATA columns : {len(CATA_cols)}")

In [ ]:
# ============================================================
# LIKERT LEVEL MAPPINGS
# Loads the full set of Likert response strings mapped to
# numeric levels. Used by map_likert_to_decimal().
# ============================================================

with open('likert_levels.json', 'r') as f:
    likert_data = json.load(f)
    level_1  = likert_data['level_1']
    level_2  = likert_data['level_2']
    level_3  = likert_data['level_3']
    level_4  = likert_data['level_4']
    level_5  = likert_data['level_5']
    level_6  = likert_data['level_6']
    level_7  = likert_data['level_7']
    level_8  = likert_data['level_8']
    level_9  = likert_data['level_9']
    level_10 = likert_data['level_10']
    level_11 = likert_data['level_11']
    level_12 = likert_data['level_12']
    level_13 = likert_data['level_13']
    level_14 = likert_data['level_14']
    level_15 = likert_data['level_15']
    level_16 = likert_data['level_16']
    level_17 = likert_data['level_17']
    level_18 = likert_data['level_18']
    level_19 = likert_data['level_19']
    level_20 = likert_data['level_20']

print('Likert level mappings loaded')

---
## SECTION 2 — Data Ingestion & Cleaning
Functions for loading, encoding detection, header cleaning, whitespace removal, column type conversion, and HDF5 intermediate storage.

In [ ]:
# ── HDF5 Intermediate Store ──────────────────────────────────────────────────
# Survey data is processed in stages. Intermediate DataFrames are saved to
# an HDF5 file for fast reloading without re-running earlier steps.
# This also provides traceability — any intermediate state can be recovered.

def save_dataframe_to_hdf5(df, key, file_path):
    """
    Save a DataFrame to HDF5 under a named key.
    If the file does not exist it is created automatically.
    If the key already exists the DataFrame is overwritten.

    Parameters:
    - df (pd.DataFrame): DataFrame to save.
    - key (str): Key name for storage (e.g. 'df_mapped_study_D3').
    - file_path (str): Path to the HDF5 file.
    """
    with pd.HDFStore(file_path) as store:
        store[key] = df
        print(f"Saved '{key}' to '{file_path}'")


def load_dataframe_from_hdf5(key, file_path):
    """
    Load a named DataFrame from HDF5.

    Parameters:
    - key (str): Key name to retrieve.
    - file_path (str): Path to the HDF5 file.

    Returns:
    - pd.DataFrame or None if key not found.
    """
    with pd.HDFStore(file_path) as store:
        if key in store:
            df = store[key]
            print(f"Loaded '{key}'")
            return df
        else:
            print(f"Key '{key}' not found in HDF5 file.")
            return None


def list_dataframes_in_hdf5(file_path):
    """
    List all DataFrame keys stored in the HDF5 file.

    Returns:
    - list of key strings.
    """
    with pd.HDFStore(file_path) as store:
        keys = store.keys()
        print("Stored DataFrames:")
        for key in keys:
            print(f"  {key}")
        return keys


def delete_dataframe_from_hdf5(file_path, key):
    """
    Delete a named DataFrame from the HDF5 file.

    Parameters:
    - file_path (str): Path to the HDF5 file.
    - key (str): Key to delete.
    """
    with pd.HDFStore(file_path) as store:
        if key in store:
            store.remove(key)
            print(f"Deleted '{key}' from '{file_path}'.")
        else:
            print(f"Key '{key}' not found.")

In [ ]:
# ── CSV Loading & Encoding ───────────────────────────────────────────────────

def detect_encoding(file_path):
    """
    Detect the character encoding of a file.
    Reads the first 5000 bytes — sufficient for most survey exports.

    Returns:
    - str: Detected encoding (e.g. 'utf-8', 'iso-8859-1').
    """
    with open(file_path, 'rb') as f:
        raw = f.read(5000)
    return chardet.detect(raw)['encoding']


def clean_csv_file(input_file, output_file, headers_to_replace, columns_to_drop):
    """
    Clean a raw survey CSV export:
      - Strips leading/trailing whitespace from headers
      - Renames headers using a replacement dictionary
      - Drops columns with no heading or in the drop list
      - Writes a cleaned copy to output_file

    Parameters:
    - input_file (str): Path to the raw CSV.
    - output_file (str): Path to write the cleaned CSV.
    - headers_to_replace (dict): {old_header: new_header} mapping.
      Commonly used for multilingual question text normalisation.
    - columns_to_drop (list): Column names to remove entirely.
    """
    with open(input_file, mode='r', newline='', encoding='utf-8-sig') as infile, \
         open(output_file, mode='w', newline='', encoding='utf-8-sig') as outfile:

        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        headers = [h.strip() for h in next(reader)]
        headers = [headers_to_replace.get(h, h) for h in headers]
        valid_headers = [h for h in headers if h and h not in columns_to_drop]
        indexes_to_keep = [i for i, h in enumerate(headers) if h in valid_headers]

        writer.writerow(valid_headers)
        for row in reader:
            writer.writerow([row[i] for i in indexes_to_keep])


def load_and_select(csv_file_path):
    """
    Load a questionnaire mapping CSV and return column names
    marked as 'ALL' in column C.

    Returns:
    - list of column name strings.
    """
    selected_text = []
    with open(csv_file_path, 'r') as file:
        for line in file:
            items = line.strip().split(',')
            if items[2] == 'ALL':
                selected_text.append(items[3])
    return selected_text


def combine_csv_files(file_paths: List[str],
                      column_mapping: Dict[str, str],
                      output_path: Optional[str] = None,
                      reset_index: bool = True) -> pd.DataFrame:
    """
    Load multiple CSV files and combine them with unified column headings.
    Useful for combining rotations from the same study.

    Parameters:
    - file_paths (List[str]): CSV files to combine.
    - column_mapping (Dict[str, str]): {raw_name: standardised_name} mapping.
    - output_path (Optional[str]): If provided, saves combined CSV here.
    - reset_index (bool): Reset the index after combining.

    Returns:
    - pd.DataFrame: Combined DataFrame with standardised columns.
    """
    combined_dfs = []
    print(f"Processing {len(file_paths)} CSV files...")

    for i, file_path in enumerate(file_paths, 1):
        try:
            if not os.path.exists(file_path):
                print(f"  ⚠️  File not found: {file_path}. Skipping.")
                continue
            df = pd.read_csv(file_path)
            print(f"  {i}. Loaded {file_path}: {len(df)} rows")
            available = [c for c in df.columns if c in column_mapping]
            if not available:
                print(f"  ⚠️  No mappable columns in {file_path}. Skipping.")
                continue
            df_subset = df[available].copy()
            df_subset.rename(columns=column_mapping, inplace=True)
            combined_dfs.append(df_subset)
        except Exception as e:
            print(f"  ❌ Error loading {file_path}: {e}")

    if not combined_dfs:
        print("❌ No valid files processed.")
        return pd.DataFrame()

    combined_df = pd.concat(combined_dfs, ignore_index=True)
    if reset_index:
        combined_df = combined_df.reset_index(drop=True)

    print(f"\n✅ Combined: {len(combined_df)} rows × {len(combined_df.columns)} columns")
    if output_path:
        combined_df.to_csv(output_path, index=False)
        print(f"💾 Saved to: {output_path}")
    return combined_df

In [ ]:
# ── DataFrame Cleaning Utilities ─────────────────────────────────────────────

def whitespace_remover(dataframe):
    """
    Strip leading and trailing whitespace from all string columns.
    Ignores NaN values to avoid type errors.
    """
    for col in dataframe.columns:
        if dataframe[col].dtype == 'object':
            dataframe[col] = dataframe[col].apply(
                lambda x: x.strip() if isinstance(x, str) else x
            )


def convert_columns_to_float(df, columns):
    """
    Convert specified columns to float, stripping hidden characters.
    Handles BOM characters and non-numeric strings gracefully.

    Parameters:
    - df (pd.DataFrame): Input DataFrame.
    - columns (list): Column names to convert.

    Returns:
    - pd.DataFrame with converted columns.
    """
    for col in columns:
        if col in df.columns:
            try:
                df[col] = df[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
                df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)
                print(f"  {col} → float")
            except ValueError:
                print(f"  {col} could not be converted.")
        else:
            print(f"  {col} not found.")
    return df


def clean_column_headers(df, words_to_remove):
    """
    Remove specified substrings from column headers using regex.
    Useful for stripping product variant names from shared questions
    (e.g. " Vanilla", " Chocolate") to produce unified column names
    across rotations.

    Parameters:
    - df (pd.DataFrame): DataFrame whose columns to clean.
    - words_to_remove (list): Substrings to remove.

    Returns:
    - pd.DataFrame with cleaned column headers.
    """
    pattern = re.compile('|'.join(re.escape(w) for w in words_to_remove))
    df.columns = [pattern.sub('', col).strip() for col in df.columns]
    return df


def full_string_replace_if_contains(df, column_name, pattern, replacement):
    """
    Replace entire cell value with a string if the cell contains a pattern.
    Used for normalising free-text customer feedback categories.

    Parameters:
    - column_name (str): Column to search.
    - pattern (str): Regex pattern to search for.
    - replacement (str): Replacement string for matching cells.
    """
    if column_name in df.columns:
        df[column_name] = np.where(
            df[column_name].str.contains(pattern, regex=True, na=False),
            replacement, df[column_name]
        )
    else:
        print(f"Column '{column_name}' not found.")
    return df


def find_and_replace_multiple_columns(df, columns, to_replace_dict):
    """
    Find and replace values across multiple columns using a dictionary.

    Parameters:
    - columns (list): Columns to apply replacements to.
    - to_replace_dict (dict): {value_to_replace: replacement}.

    Returns:
    - pd.DataFrame with values replaced.
    """
    for col in columns:
        if col in df.columns:
            for to_replace, replacement in to_replace_dict.items():
                for item in to_replace:
                    df[col] = df[col].replace(item, replacement)
        else:
            print(f"Column '{col}' not found.")
    return df


def replace_nans_with_zero(df, columns):
    """
    Replace NaN with 0 in specified columns.
    Used before dichotomisation to ensure binary encoding is complete.
    """
    for col in columns:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not in DataFrame.")
    df[columns] = df[columns].fillna(0)
    return df


def sanitize_filename(filename):
    """
    Remove characters invalid for filenames from a string.
    Used when generating output filenames from question text.
    """
    return re.sub(r'[<>:"/\\|?*]', '_', filename)


def clean_sheet_name(sheet_name):
    """
    Sanitise an Excel sheet name.
    Removes invalid characters and truncates to 31 characters.
    """
    sheet_name = re.sub(r'[\\/*?[\]:]', '_', sheet_name)
    return sheet_name[:31]


def convert_columns_dtype(df, columns, dtype):
    """Convert specified columns to a given dtype."""
    df[columns] = df[columns].astype(dtype)
    return df


def sort_by_customer_id(df):
    """
    Sort a DataFrame by 'Customer ID' and reset the index.
    Required before paired statistical tests (Wilcoxon, McNemar)
    to ensure respondents are aligned across product DataFrames.
    """
    if 'Customer ID' not in df.columns:
        raise ValueError("DataFrame must contain a 'Customer ID' column.")
    return df.sort_values(by='Customer ID', ascending=True).reset_index(drop=True)


def sort_dataframes_by_column(dfs, sort_column):
    """
    Sort multiple DataFrames by the same column and validate alignment.
    Raises ValueError if Customer ID columns do not match after sorting.

    Parameters:
    - dfs (list of pd.DataFrame): DataFrames to sort.
    - sort_column (str): Column to sort by.

    Returns:
    - list of sorted DataFrames.
    """
    sorted_dfs = [df.sort_values(by=sort_column).reset_index(drop=True) for df in dfs]
    for i in range(1, len(sorted_dfs)):
        if not sorted_dfs[i][sort_column].equals(sorted_dfs[i-1][sort_column]):
            raise ValueError("Customer ID columns do not match after sorting.")
    return sorted_dfs


def rename_columns_with_dict(df, rename_dict):
    """
    Rename columns using a dictionary. Returns the list of new names applied.
    Used for abbreviated column names in JAR/driver analysis output.

    Returns:
    - list of new column name strings that were applied.
    """
    cols_to_rename = {c: rename_dict[c] for c in df.columns if c in rename_dict}
    df.rename(columns=cols_to_rename, inplace=True)
    return list(cols_to_rename.values())


def get_df_name(df):
    """Return the variable name of a DataFrame from the global namespace."""
    for name, obj in globals().items():
        if obj is df:
            return name

In [ ]:
# ── Respondent Splitting ─────────────────────────────────────────────────────
# Survey rotations assign respondents to product sequences via email address
# prefixes or lists. These functions split the full dataset into per-rotation
# subsets for paired statistical comparison.

def split_dataframe_by_email_list(df, email_list1, email_list2, email_column='Email'):
    """
    Split a DataFrame into two subsets by email address list membership.

    Returns:
    - tuple (df1, df2): DataFrames filtered by each email list.
    """
    df1 = df[df[email_column].isin(email_list1)].copy()
    df2 = df[df[email_column].isin(email_list2)].copy()
    return df1, df2


def split_dataframe_by_email(df):
    """
    Split a DataFrame by whether the Email column starts with '1' or '2'.
    Used in studies where rotation is encoded in the email prefix.

    Returns:
    - tuple (df_1, df_2).
    """
    df_1 = df[df['Email'].str.startswith('1')]
    df_2 = df[df['Email'].str.startswith('2')]
    return df_1, df_2


def split_dataframe_by_email_var(df, char_1, char_2):
    """
    Split a DataFrame by the first character of the Email column.

    Parameters:
    - char_1 (str): First character to filter on.
    - char_2 (str): Second character to filter on.

    Returns:
    - tuple (df_1, df_2).
    """
    df['Email'] = df['Email'].str.lower()
    df_1 = df[df['Email'].str.startswith(char_1)]
    df_2 = df[df['Email'].str.startswith(char_2)]
    return df_1, df_2

---
## SECTION 3 — Data Reshaping & Summarisation
Likert mapping, scale inversion, Top/Bottom Box calculations, crosstabs, and dichotomisation.

In [ ]:
# ── Likert Scale Mapping ─────────────────────────────────────────────────────

def map_likert_to_decimal(response):
    """
    Map a Likert response string to its numeric category level.
    Levels 1–20 are defined in likert_levels.json (loaded in Section 1).
    Returns None for unrecognised values.
    """
    if isinstance(response, str):
        response = response.strip()
    levels = [
        level_1, level_2, level_3, level_4, level_5,
        level_6, level_7, level_8, level_9, level_10,
        level_11, level_12, level_13, level_14, level_15,
        level_16, level_17, level_18, level_19, level_20
    ]
    for i, level in enumerate(levels, 1):
        if response in level:
            return float(i)
    return None


def apply_mapping_to_columns(df, columns_to_skip):
    """
    Apply map_likert_to_decimal to all columns not in the skip list.
    Skipped columns are typically ID, email, and open-ended text columns.

    Parameters:
    - columns_to_skip (list): Column names to leave unchanged.

    Returns:
    - pd.DataFrame with Likert columns converted to numeric.
    """
    for col in df.columns:
        if col not in columns_to_skip:
            df[col] = df[col].apply(map_likert_to_decimal)
    return df


def invert_10pt_likert_scale(df, columns):
    """
    Invert 10-point Likert responses (1→10, 2→9, ..., 10→1).
    Required for negatively-worded attributes (e.g. 'level of foam',
    'amount of lumps') where a lower raw score indicates better quality.

    Parameters:
    - columns (list): Columns to invert.

    Returns:
    - pd.DataFrame (copy) with specified columns inverted.
    """
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found.")
        df[col] = df[col].apply(lambda x: 11 - x if pd.notnull(x) else x)
    return df


def convert_10_to_5(df, columns):
    """
    Map 10-point scale responses to a 5-point equivalent.
    Mapping: {1,2}→1, {3,4}→2, {5,6}→3, {7,8}→4, {9,10}→5.
    Used before Spearman correlation and Condition Index calculations.

    Parameters:
    - columns (list): 10-point columns to convert.

    Returns:
    - pd.DataFrame with converted columns (in-place).
    """
    mapping = {1:1, 2:1, 3:2, 4:2, 5:3, 6:3, 7:4, 8:4, 9:5, 10:5}
    if not columns:
        return df
    for col in columns:
        try:
            df.loc[:, col] = df[col].replace(mapping)
        except Exception as e:
            print(f"Skipping '{col}': {e}")
    return df


def summarize_likert_counts(df, columns):
    """
    Count responses per Likert level for specified columns.

    Returns:
    - tuple (summary_df, summary_percentage_df): Raw counts and percentages.
    """
    summary = {col: df[col].value_counts().sort_index() for col in columns}
    summary_df = pd.DataFrame(summary).fillna(0)
    summary_df.loc['Total'] = summary_df.sum()
    summary_percentage_df = (summary_df.div(summary_df.loc['Total'], axis=1) * 100).round(0)
    return summary_df, summary_percentage_df

In [ ]:
# ── Top/Bottom Box Summarisation ─────────────────────────────────────────────
# Core reporting function. Appends count, percentage, and box-score summary
# rows to the bottom of the response DataFrame. Used as the primary output
# table in every study report.
#
# Box score definitions:
#   Bottom Box  = % scoring 1
#   % B2B       = % scoring 1–2
#   % B4B       = % scoring 1–4
#   % Middle    = % scoring 3
#   % T2B       = % scoring 4–5  (5-point scale)
#   % T4B       = % scoring 7–10 (10-point scale)
#   Top Box     = % scoring 5
#   % TT2B      = % scoring 9–10 (10-point scale)
#   Mean        = arithmetic mean of 10-point columns

def summarize_and_append(df, columns_to_skip, columns_to_summarize, TEN):
    """
    Summarise Likert columns and append box scores to the DataFrame.

    Parameters:
    - df (pd.DataFrame): Response DataFrame.
    - columns_to_skip (list): Columns to exclude from summarisation.
    - columns_to_summarize (list): 5-point columns to include in box scores.
    - TEN (list): 10-point columns to include in box scores and Mean row.

    Returns:
    - pd.DataFrame with summary rows appended.
    """
    summary = {}
    all_cols_to_summarize = columns_to_summarize + TEN

    for col in df.columns:
        if col not in columns_to_skip:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                summary[col] = df[col].value_counts().sort_index()
            except Exception as e:
                print(f"  Error processing '{col}': {e}")

    if not summary:
        return df

    summary_df = pd.DataFrame(summary).fillna(0)
    summary_df.loc['Total'] = summary_df.sum()
    summary_percentage_df = summary_df.div(summary_df.loc['Total'], axis=1) * 100

    for col in df.columns:
        if col not in summary_df.columns:
            summary_df[col] = 0
            summary_percentage_df[col] = 0

    summary_df = summary_df[df.columns]
    summary_percentage_df = summary_percentage_df[df.columns]
    summary_df.index = ['Summary_' + str(i) for i in summary_df.index]
    summary_percentage_df.index = ['Percentage_' + str(i) for i in summary_percentage_df.index]

    def pct(label, col):
        return summary_percentage_df.loc[label, col] if label in summary_percentage_df.index else 0

    custom_summary = {k: [] for k in [
        'Bottom Box', '% B2B', '% B4B', '% Middle',
        '% T2B', '% T3B', '% T4B', 'Top Box', '% TT2B', '% TB2B'
    ]}

    for col in all_cols_to_summarize:
        custom_summary['Bottom Box'].append(pct('Percentage_1.0', col))
        custom_summary['% B2B'].append(pct('Percentage_1.0', col) + pct('Percentage_2.0', col))
        custom_summary['% B4B'].append(sum(pct(f'Percentage_{i}.0', col) for i in range(1, 5)))
        custom_summary['% Middle'].append(pct('Percentage_3.0', col))
        custom_summary['% T2B'].append(pct('Percentage_4.0', col) + pct('Percentage_5.0', col))
        custom_summary['% T3B'].append(sum(pct(f'Percentage_{i}.0', col) for i in range(3, 6)))
        custom_summary['% T4B'].append(sum(pct(f'Percentage_{i}.0', col) for i in range(7, 11)))
        custom_summary['Top Box'].append(pct('Percentage_5.0', col))
        custom_summary['% TT2B'].append(pct('Percentage_9.0', col) + pct('Percentage_10.0', col))
        custom_summary['% TB2B'].append(pct('Percentage_1.0', col) + pct('Percentage_2.0', col))

    custom_summary_df = pd.DataFrame(custom_summary, index=all_cols_to_summarize).T
    df_with_summary = pd.concat([df, summary_df, summary_percentage_df, custom_summary_df], sort=False)

    ten_point_means = df[TEN].mean()
    ten_point_means.name = 'Mean'
    df_with_summary = pd.concat([df_with_summary, ten_point_means.to_frame().T], sort=False)
    df_with_summary.at['Mean', 'Customer ID'] = 'Mean'

    return df_with_summary

In [ ]:
# ── Crosstabs & Dichotomisation ──────────────────────────────────────────────

def create_crosstabs_and_save_to_excel(df, column_pairs_dict, output_file):
    """
    Create pairwise crosstabs for specified column pairs and save to Excel.
    Each crosstab is written to a separate sheet.

    Parameters:
    - column_pairs_dict (dict): {name: [col1, col2]} pairs to cross-tabulate.
    - output_file (str): Output Excel file path.
    """
    crosstab_dict = {}
    for name, columns in column_pairs_dict.items():
        if len(columns) != 2:
            print(f"Skipping '{name}': needs exactly 2 columns.")
            continue
        col1, col2 = columns
        if col1 in df.columns and col2 in df.columns:
            crosstab_dict[name] = pd.crosstab(df[col1], df[col2])
        else:
            print(f"Skipping '{name}': column(s) not found.")

    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        for name, table in crosstab_dict.items():
            table.to_excel(writer, sheet_name=name)
    print(f"Crosstabs saved to {output_file}")


def dichotomize_columns(df, columns, output_file, dichot_list):
    """
    Dichotomise Likert columns to binary (1/0) based on a value list.
    Values in dichot_list → 1; all other non-missing values → 0;
    NaN remains NaN.

    Typical use: Top-2-Box dichotomisation for McNemar's test.
    dichot_list = [4, 5] produces a T2B binary column.

    Parameters:
    - columns (list): Columns to dichotomise.
    - output_file (str): Excel file to save the dichotomised output.
    - dichot_list (list): Values to code as 1.

    Returns:
    - pd.DataFrame of dichotomised columns.
    """
    dichotomized_df = pd.DataFrame()
    print(f"Dichotomising with values: {dichot_list}")
    for col in columns:
        if col in df.columns:
            dichotomized_df[col] = df[col].apply(
                lambda x: 1 if x in dichot_list else (0 if pd.notna(x) else pd.NA)
            )
        else:
            print(f"  Warning: '{col}' not found.")

    dichotomized_df.to_excel(output_file, index=False)
    print(f"Saved to {output_file}")
    return dichotomized_df


def summary_count_with_percentage_and_save(df, columns, output_file):
    """
    Summarise unique value counts and percentages for specified columns.
    Writes all column summaries side-by-side in a single Excel sheet.

    Parameters:
    - columns (list): Columns to summarise.
    - output_file (str): Output Excel file path.
    """
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        workbook  = writer.book
        worksheet = workbook.add_worksheet('Combined_Summary')
        writer.sheets['Combined_Summary'] = worksheet
        col_start = 0
        for col_name in columns:
            if col_name not in df.columns:
                raise ValueError(f"Column '{col_name}' not found.")
            count_series = df[col_name].value_counts(dropna=False)
            pct_series   = (count_series / count_series.sum()) * 100
            summary_df   = pd.DataFrame({
                col_name: count_series.index,
                'Count': count_series.values,
                '%': pct_series.values
            })
            summary_df.to_excel(writer, sheet_name='Combined_Summary',
                                startrow=0, startcol=col_start, index=False)
            col_start += len(summary_df.columns) + 1
    print(f"Summaries saved to {output_file}")


def summary_count_across_dataframes_and_save(dataframes, columns, output_file):
    """
    Combine multiple DataFrames and produce a pooled summary count per column.
    Writes results side-by-side in a single Excel sheet.
    Used for combined product-level or market-level summaries.

    Parameters:
    - dataframes (list of pd.DataFrame): DataFrames to pool.
    - columns (list): Columns to summarise.
    - output_file (str): Output Excel file path.
    """
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        workbook  = writer.book
        worksheet = workbook.add_worksheet('Combined_Summary')
        writer.sheets['Combined_Summary'] = worksheet
        col_start = 0
        for col_name in columns:
            combined = pd.concat([df[col_name] for df in dataframes], ignore_index=True)
            count_series = combined.value_counts(dropna=False)
            pct_series   = count_series / count_series.sum()
            summary_df   = pd.DataFrame({
                col_name: count_series.index,
                'Count': count_series.values,
                '%': pct_series.values
            })
            summary_df.loc[len(summary_df)] = ['Total', count_series.sum(), 1.0]
            summary_df.to_excel(writer, sheet_name='Combined_Summary',
                                startrow=0, startcol=col_start, index=False)
            pct_fmt = workbook.add_format({'num_format': '0.0%'})
            worksheet.set_column(col_start + 2, col_start + 2, 10, pct_fmt)
            col_start += len(summary_df.columns) + 1
    print(f"Pooled summaries saved to {output_file}")


def summary_count_with_spell_correction_and_save(df, columns, output_file):
    """
    Summarise comma-separated CATA-style open fields with spell correction.
    Useful for free-text CATA where respondents type their own attributes
    with inconsistent spelling.

    Parameters:
    - columns (list): Columns containing comma-separated text.
    - output_file (str): Output Excel file path.
    """
    def clean_and_correct(text):
        if pd.isna(text):
            return []
        items = [item.strip().lower() for item in str(text).split(',')]
        return [str(TextBlob(word).correct()) for word in items if word]

    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        workbook  = writer.book
        worksheet = workbook.add_worksheet('Cleaned_Summary')
        writer.sheets['Cleaned_Summary'] = worksheet
        col_start = 0
        for col_name in columns:
            if col_name not in df.columns:
                raise ValueError(f"Column '{col_name}' not found.")
            all_terms = []
            for val in df[col_name]:
                all_terms.extend(clean_and_correct(val))
            counter = Counter(all_terms)
            total   = sum(counter.values())
            summary_df = pd.DataFrame({
                col_name: list(counter.keys()),
                'Count': list(counter.values()),
                '%': [round(c / total * 100, 2) for c in counter.values()]
            }).sort_values('%', ascending=False)
            summary_df.to_excel(writer, sheet_name='Cleaned_Summary',
                                startrow=0, startcol=col_start, index=False)
            col_start += len(summary_df.columns) + 1
    print(f"Spell-corrected CATA summaries saved to {output_file}")


def transpose_columns_to_rows(storage_df, df, columns_to_transpose, save_name, extract):
    """
    Extract a specific row value per column from df and store it in
    storage_df as a new column. Used to build the 1-pager comparison table
    where each product becomes a column and each question becomes a row.

    Parameters:
    - storage_df (pd.DataFrame): Accumulator DataFrame.
    - df (pd.DataFrame): Source DataFrame (with summary rows appended).
    - columns_to_transpose (list): Columns to extract from df.
    - save_name (str): Column name to add to storage_df.
    - extract (str): Row label in df to extract (e.g. '% T2B').

    Returns:
    - pd.DataFrame: Updated storage_df.
    """
    if save_name not in storage_df.columns:
        storage_df[save_name] = None
    for col in columns_to_transpose:
        if col not in storage_df['comparison'].values:
            storage_df = pd.concat(
                [storage_df, pd.DataFrame([{'comparison': col, save_name: None}])],
                ignore_index=True
            )
        if col in df.columns:
            value = df.loc[extract, col]
            storage_df.loc[storage_df['comparison'] == col, save_name] = value
    return storage_df


def copy_index_to_column(df, index_list):
    """
    Copy index names into the 'Customer ID' column for summary rows.
    Ensures summary rows are labelled correctly in the output Excel.

    Parameters:
    - index_list (list): Row index names to label.
    """
    for idx in df.index:
        if idx in index_list:
            df.at[idx, 'Customer ID'] = idx

---
## SECTION 4 — Statistical Testing
Non-parametric tests for pairwise product comparison. All tests support multiple DataFrames and return structured result dictionaries.

In [ ]:
# ── Effect Size Helpers ──────────────────────────────────────────────────────

def cliffs_delta_MWU(data1, data2):
    """
    Calculate Cliff's Delta effect size for two groups.
    Measures the probability that a value from group1 exceeds a value
    from group2, accounting for direction.

    Interpretation thresholds (Romano et al., 2006):
      |δ| < 0.147  → negligible
      |δ| < 0.330  → small
      |δ| < 0.474  → medium
      |δ| ≥ 0.474  → large

    Returns:
    - str: Effect size label ('negligible', 'small', 'medium', 'large').
    """
    from itertools import product as iproduct

    group1 = np.array(data1.dropna().values if hasattr(data1, 'dropna') else data1)
    group2 = np.array(data2.dropna().values if hasattr(data2, 'dropna') else data2)
    group1 = group1[~np.isnan(group1)]
    group2 = group2[~np.isnan(group2)]

    if len(group1) == 0 or len(group2) == 0:
        return 'insufficient data'

    more  = sum(x > y for x, y in iproduct(group1, group2))
    less  = sum(x < y for x, y in iproduct(group1, group2))
    delta = (more - less) / (len(group1) * len(group2))

    abs_delta = abs(delta)
    if abs_delta < 0.147: return 'negligible'
    if abs_delta < 0.330: return 'small'
    if abs_delta < 0.474: return 'medium'
    return 'large'

In [ ]:
# ── Mann-Whitney U Test ──────────────────────────────────────────────────────
# Non-parametric test for unpaired ordinal data (10-point Likert scale).
# Used for comparing 10-point scale columns between product groups.
# Returns results at both 90% and 95% confidence levels.

def mann_whitney_u_test_for_multiple_dfs(dfs, df_names, questions):
    """
    Perform pairwise Mann-Whitney U tests across multiple DataFrames.
    Includes Cliff's Delta and r-from-Z effect sizes.

    Parameters:
    - dfs (list of pd.DataFrame): DataFrames to compare.
    - df_names (list of str): Labels for each DataFrame.
    - questions (list of str): Column names to test.

    Returns:
    - dict: {question: {comparison: {U-stat, P-value, Strength, ...}}}
    """
    all_results = {}
    for question in questions:
        question_results = {}
        for i in range(len(dfs)):
            for j in range(i + 1, len(dfs)):
                name1, name2 = df_names[i], df_names[j]
                data1 = dfs[i][question].dropna().astype(float)
                data2 = dfs[j][question].dropna().astype(float)

                if len(data1) == 0 or len(data2) == 0:
                    question_results[f"{name1} vs {name2}"] = {
                        'U-statistic': np.nan, 'P-value': np.nan,
                        'Strength': 'N/A', 'r_Z_strength': 'N/A',
                        'Hypothesis at 95%': 'Not enough data',
                        'Hypothesis at 90%': 'Not enough data'
                    }
                    continue

                u_stat, p_val = mannwhitneyu(data1, data2, alternative='two-sided')
                strength      = cliffs_delta_MWU(data1, data2)

                n1, n2  = len(data1), len(data2)
                u_mean  = n1 * n2 / 2
                u_std   = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
                z       = (u_stat - u_mean) / u_std
                r       = abs(z) / np.sqrt(n1 + n2)
                r_str   = 'negligible' if r < 0.1 else 'small' if r < 0.3 else 'medium' if r < 0.5 else 'large'

                question_results[f"{name1} vs {name2}"] = {
                    'U-statistic': u_stat,
                    'P-value': p_val,
                    'Strength': strength,
                    'r_Z_strength': r_str,
                    'Hypothesis at 95%': 'Reject null => Significant difference' if p_val < 0.05 else 'Accept null => No significant difference',
                    'Hypothesis at 90%': 'Reject null => Significant difference' if p_val < 0.10 else 'Accept null => No significant difference'
                }
        all_results[question] = question_results
    return all_results

In [ ]:
# ── McNemar's Test ───────────────────────────────────────────────────────────
# Paired test for binary (dichotomised) data comparing two products
# evaluated by the same respondents. Used after Top-2-Box dichotomisation.
#
# Implementation notes:
#   - Uses chi-squared approximation when b+c ≥ 25
#   - Falls back to exact binomial (statsmodels) when b+c < 25
#   - 'E' flag in output indicates exact test was used
#   - Guards against length mismatch, non-2×2 tables, and zero discordant pairs

def mcnemar_test_on_dataframes(dfs, df_names, columns):
    """
    Perform pairwise McNemar's tests across multiple DataFrames.

    Parameters:
    - dfs (list of pd.DataFrame): Dichotomised DataFrames to compare.
    - df_names (list of str): Labels for each DataFrame.
    - columns (list of str): Binary columns to test.

    Returns:
    - dict: {column: {comparison: {95% Confidence, 90% Confidence, P-value, chi2, b, c, e}}}
    """
    all_results = {}

    for col in columns:
        column_results = {}
        for i in range(len(dfs)):
            for j in range(i + 1, len(dfs)):
                name1, name2 = df_names[i], df_names[j]
                df1 = dfs[i].reset_index(drop=True)
                df2 = dfs[j].reset_index(drop=True)
                comp = f"{name1} vs {name2}"
                e = ''

                try:
                    if col not in df1.columns or col not in df2.columns:
                        column_results[comp] = {'95% Confidence': 'skip (column missing)', '90% Confidence': 'skip (column missing)', 'P-value': 'skip', 'chi2': None, 'b': None, 'c': None, 'e': e}
                        continue

                    s1, s2 = df1[col], df2[col]
                    if len(s1) != len(s2):
                        column_results[comp] = {'95% Confidence': 'skip (length mismatch)', '90% Confidence': 'skip (length mismatch)', 'P-value': 'skip', 'chi2': None, 'b': None, 'c': None, 'e': e}
                        continue

                    ct = pd.crosstab(s1, s2)
                    if ct.shape != (2, 2):
                        column_results[comp] = {'95% Confidence': 'skip (not 2×2)', '90% Confidence': 'skip (not 2×2)', 'P-value': 'skip', 'chi2': None, 'b': None, 'c': None, 'e': e}
                        continue

                    b, c = int(ct.iloc[0, 1]), int(ct.iloc[1, 0])
                    if (b + c) == 0:
                        column_results[comp] = {'95% Confidence': 'no discordant pairs', '90% Confidence': 'no discordant pairs', 'P-value': 'skip', 'chi2': 0.0, 'b': b, 'c': c, 'e': e}
                        continue

                    if (b + c) >= 25:
                        stat    = round(((b - c) ** 2) / (b + c), 4)
                        p_value = round(1 - chi2.cdf(stat, 1), 4)
                    else:
                        result  = mcnemar(ct, exact=True)
                        p_value = round(float(result.pvalue), 4)
                        stat    = round(((b - c) ** 2) / (b + c), 4)
                        e       = 'E'

                    column_results[comp] = {
                        '95% Confidence': 'Reject null => Significant difference' if p_value < 0.05 else 'Accept null => No significant difference',
                        '90% Confidence': 'Reject null => Significant difference' if p_value < 0.10 else 'Accept null => No significant difference',
                        'P-value': p_value, 'chi2': stat, 'b': b, 'c': c, 'e': e
                    }

                except Exception as ex:
                    print(f"  Error for '{col}' between '{name1}' and '{name2}': {ex}")

        if column_results:
            all_results[col] = column_results
    return all_results

In [ ]:
# ── Wilcoxon Signed-Rank Test ─────────────────────────────────────────────────
# Paired test for continuous ordinal data (10-point scale).
# Assumes the same respondents evaluated both products.
# More powerful than Mann-Whitney U for paired designs.

def wilcoxon_test_multiple_dataframes_named(dataframes, columns, dataframe_names, verbose=True):
    """
    Perform paired Wilcoxon Signed-Rank tests across multiple DataFrames.
    Aligns respondents by position — sort DataFrames by Customer ID first.

    Parameters:
    - dataframes (list of pd.DataFrame): DataFrames to compare.
    - columns (list of str): Columns to test.
    - dataframe_names (list of str): Labels for each DataFrame.
    - verbose (bool): Print errors if encountered.

    Returns:
    - dict: {column: {comparison: {Wilcoxon Statistic, P-value, Z value, Intensity, ...}}}
    """
    try:
        all_results = {}
        for col in columns:
            question_results = {}
            if all(col in df.columns for df in dataframes):
                for i in range(len(dataframes)):
                    for j in range(i + 1, len(dataframes)):
                        name1, name2 = dataframe_names[i], dataframe_names[j]
                        df1, df2 = dataframes[i], dataframes[j]
                        df1[col] = pd.to_numeric(df1[col], errors='coerce')
                        df2[col] = pd.to_numeric(df2[col], errors='coerce')
                        paired = pd.concat(
                            [df1[col].reset_index(drop=True), df2[col].reset_index(drop=True)], axis=1
                        ).dropna()
                        if paired.empty:
                            continue

                        stat, p_val = stats.wilcoxon(paired.iloc[:, 0], paired.iloc[:, 1])
                        z_value    = abs(stats.norm.ppf(p_val / 2))
                        n          = len(paired)
                        r          = z_value / (n ** 0.5)
                        intensity  = 'large' if r > 0.5 else 'moderate' if r > 0.3 else 'small' if r >= 0.1 else 'undetermined'

                        question_results[f"{name1} vs {name2}"] = {
                            'Wilcoxon Statistic': stat,
                            'P-value': p_val,
                            'Z value': z_value,
                            'Intensity': intensity,
                            'Hypothesis at 95%': 'Reject null => Significant difference' if p_val < 0.05 else 'Accept null => No significant difference',
                            'Hypothesis at 90%': 'Reject null => Significant difference' if p_val < 0.10 else 'Accept null => No significant difference'
                        }
            all_results[col] = question_results
        return all_results
    except Exception as e:
        if verbose:
            print(f"Error in Wilcoxon function: {e}")
        return {}


# ── Fisher's Exact Test ───────────────────────────────────────────────────────
# Used for binary data when sample sizes are too small for chi-squared.
# Typically applied to CATA binary presence/absence data.

def fishers_exact_test_for_multiple_dfs(dfs, df_names, questions):
    """
    Perform pairwise Fisher's Exact tests across multiple DataFrames.
    Only applicable to binary (2-category) columns.

    Parameters:
    - dfs (list of pd.DataFrame): DataFrames to compare.
    - df_names (list of str): Labels for each DataFrame.
    - questions (list of str): Binary columns to test.

    Returns:
    - dict: {question: {comparison: {P-value, Hypothesis at 95%, Hypothesis at 90%}}}
    """
    all_results = {}
    for question in questions:
        question_results = {}
        for i in range(len(dfs)):
            for j in range(i + 1, len(dfs)):
                name1, name2 = df_names[i], df_names[j]
                df1_counts   = dfs[i][question].value_counts()
                df2_counts   = dfs[j][question].value_counts()
                categories   = sorted(
                    set(dfs[i][question].dropna().unique()) |
                    set(dfs[j][question].dropna().unique())
                )
                ct = [[df1_counts.get(c, 0) for c in categories],
                      [df2_counts.get(c, 0) for c in categories]]

                if len(categories) == 2:
                    table = [[ct[0][0], ct[1][0]], [ct[0][1], ct[1][1]]]
                    _, p_val = fisher_exact(table, alternative='two-sided')
                    question_results[f"{name1} vs {name2}"] = {
                        'P-value': p_val,
                        'Hypothesis at 95%': 'Reject null => Significant difference' if p_val < 0.05 else 'Accept null => No significant difference',
                        'Hypothesis at 90%': 'Reject null => Significant difference' if p_val < 0.10 else 'Accept null => No significant difference'
                    }
                else:
                    question_results[f"{name1} vs {name2}"] = {
                        'Error': 'Not a 2×2 table. Fisher Exact not applicable.'
                    }
        all_results[question] = question_results
    return all_results

In [ ]:
# ── Convenience Wrappers ─────────────────────────────────────────────────────

def generate_manwhit_mcnemar_stats(df_list, analysis_cols, stats_name):
    """
    Load DataFrames from HDF5 and run Mann-Whitney U + McNemar tests.
    Prints results to console. Used as a quick batch stats function.

    Parameters:
    - df_list (list of str): HDF5 keys to load.
    - analysis_cols (list of str): Columns to test.
    - stats_name (str): Label for the console output.

    Returns:
    - tuple (mwu_results, mcnemar_results).
    """
    list_of_dfs = []
    with pd.HDFStore(h5_file_path, mode='r') as store:
        for key in df_list:
            list_of_dfs.append(store.get(key))

    results  = mann_whitney_u_test_for_multiple_dfs(list_of_dfs, df_list, analysis_cols)
    results2 = mcnemar_test_on_dataframes(list_of_dfs, df_list, analysis_cols)

    print(f"\n{stats_name} — Mann-Whitney U")
    for question, res in results.items():
        print(f"  {question}")
        for comp, vals in res.items():
            print(f"    {comp}: U={vals['U-statistic']:.1f}  p={vals['P-value']:.4f}  "
                  f"δ={vals['Strength']}  r={vals['r_Z_strength']}")
            print(f"    95%: {vals['Hypothesis at 95%']}")

    print(f"\n{stats_name} — McNemar")
    for col, comps in results2.items():
        print(f"  {col}")
        for comp, s in comps.items():
            print(f"    {comp}: p={s['P-value']}  χ²={s['chi2']}  b={s['b']}  c={s['c']}  {s['e']}")
            print(f"    95%: {s['95% Confidence']}")

    return results, results2


def generate_wilcoxon_stats(df_list, analysis_cols, stats_name):
    """
    Load DataFrames from HDF5 and run Wilcoxon Signed-Rank tests.
    Prints results to console.

    Parameters:
    - df_list (list of str): HDF5 keys to load.
    - analysis_cols (list of str): Columns to test.
    - stats_name (str): Label for the console output.

    Returns:
    - dict: Wilcoxon results.
    """
    list_of_dfs = []
    with pd.HDFStore(h5_file_path, mode='r') as store:
        for key in df_list:
            list_of_dfs.append(store.get(key))

    results = wilcoxon_test_multiple_dataframes_named(list_of_dfs, analysis_cols, df_list)

    print(f"\n{stats_name} — Wilcoxon 10-Point")
    for question, comps in results.items():
        print(f"  {question}")
        for comp, result in comps.items():
            print(f"    {comp}: p={result['P-value']:.4f}  intensity={result['Intensity']}")
            print(f"    95%: {result['Hypothesis at 95%']}")

    return results

---
## SECTION 5 — Sensory Analysis
JAR Penalty Analysis, CATA Analysis, Condition Index, Spearman Correlation, and MaxDiff.

In [ ]:
# ── Spearman Rank Correlation Heatmap ────────────────────────────────────────

def spearman_rank_correlation(df, sr_save_name, columns, width=100):
    """
    Compute and plot a Spearman Rank Correlation heatmap for specified columns.
    Saves the heatmap as a high-resolution PNG.

    Parameters:
    - df (pd.DataFrame): DataFrame containing the columns.
    - sr_save_name (str): File name suffix for the saved plot.
    - columns (list): Columns to include in the correlation matrix.
    - width (int): Max character width for column label wrapping.

    Returns:
    - pd.DataFrame: Spearman correlation matrix.
    """
    spearman_corr = df[columns].corr(method='spearman')
    wrapped = [textwrap.fill(col, width) for col in spearman_corr.columns]
    spearman_corr.columns = wrapped
    spearman_corr.index   = wrapped

    plt.figure(figsize=(35, 35))
    sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt='.2f',
                linewidths=0.5, annot_kws={'size': 10})
    plt.title(f'Spearman Rank Correlation — {sr_save_name}', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{path}Spearman_{sr_save_name}.png", dpi=400, bbox_inches='tight')
    plt.show()
    return spearman_corr

In [ ]:
# ── Condition Index & Variance Decomposition ──────────────────────────────────
# Multicollinearity diagnostic for driver analysis predictor sets.
# High Condition Index (>30) with high Variance Decomposition Proportions (>0.5)
# indicates problematic multicollinearity between predictors.
# Run before driver analysis to identify and remove collinear predictors.

def calculate_condition_index_and_variance_decomposition_v2(
        df, cols_10pt, cols_5pt, save_name, missing_threshold, verbose=False):
    """
    Calculate Condition Index and Variance Decomposition Proportions
    for multicollinearity diagnostics.

    Parameters:
    - df (pd.DataFrame): Dataset.
    - cols_10pt (list): 10-point scale predictors.
    - cols_5pt (list): 5-point scale predictors.
    - save_name (str): Suffix for the Excel output filename.
    - missing_threshold (float): Max allowed missing proportion per column.
    - verbose (bool): Print dropped columns and remaining row counts.

    Returns:
    - pd.DataFrame: Condition Index and Variance Decomposition table.
    """
    all_cols   = cols_10pt + cols_5pt
    valid_cols = [c for c in all_cols if df[c].isnull().mean() <= missing_threshold]
    dropped    = set(all_cols) - set(valid_cols)

    if verbose and dropped:
        print(f"Dropped (missingness > {missing_threshold*100:.1f}%): {list(dropped)}")

    X = df[valid_cols].dropna()
    if verbose:
        print(f"Remaining predictors: {len(valid_cols)}  Rows: {X.shape[0]}")

    if X.empty:
        return pd.DataFrame()

    X_scaled = StandardScaler().fit_transform(X)
    corr_matrix = np.corrcoef(X_scaled, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eig(corr_matrix)

    idx = eigenvalues.argsort()[::-1]
    eigenvalues  = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    condition_indices = np.sqrt(eigenvalues[0] / eigenvalues)
    Phi = eigenvectors
    n   = len(valid_cols)
    vd  = np.zeros((n, len(eigenvalues)))

    for i in range(len(eigenvalues)):
        for j in range(n):
            vd[j, i] = (Phi[j, i] ** 2) / eigenvalues[i]

    vd = np.abs(vd)
    vd = vd / vd.sum(axis=1, keepdims=True)

    result_df = pd.concat([
        pd.DataFrame({'Condition Index': condition_indices}),
        pd.DataFrame(vd.T, columns=valid_cols)
    ], axis=1)

    save_file = f"{path}CI_VD_{save_name}.xlsx"
    result_df.to_excel(save_file, index=False)
    print(f"Saved: {save_file}")
    return result_df

In [ ]:
# ── JAR Penalty Analysis ──────────────────────────────────────────────────────
# Penalty Analysis measures the impact on overall liking when a JAR attribute
# is scored as Too Low or Too High vs Just About Right.
#
# This implementation handles inverted JAR scales where:
#   1–2 = Too Much  (e.g. too sweet, too strong)
#   3   = JAR
#   4–5 = Too Little
# Standard (non-inverted) scale: 1–2 = Too Little, 3 = JAR, 4–5 = Too Much.

def penalty_analysis_invert_scale(df, jar_columns, target_column,
                                   abbreviation_dict, pa_save_name='penalty_analysis',
                                   threshold_pct=20):
    """
    Run JAR Penalty Analysis for an inverted 5-point JAR scale.
    Generates three charts:
      1. Scatter: Mean Drop vs % Respondents (Too Low and Too High)
      2. Bar chart: Mean Drop per attribute
      3. Bar chart: Penalty per attribute

    Parameters:
    - df (pd.DataFrame): Dataset containing JAR and target columns.
    - jar_columns (list): JAR attribute columns.
    - target_column (str): Overall liking or opinion column.
    - abbreviation_dict (dict): {full_col_name: abbreviated_label}.
    - pa_save_name (str): Filename base for outputs.
    - threshold_pct (float): % threshold line for the scatter chart.

    Outputs:
    - Excel file with penalty analysis results.
    - Three PNG charts.
    """
    results = []
    for jar_col in jar_columns:
        too_low  = df[df[jar_col].isin([4, 5])][target_column]
        jar      = df[df[jar_col] == 3][target_column]
        too_high = df[df[jar_col].isin([1, 2])][target_column]
        non_jar  = pd.concat([too_low, too_high])

        mean_jar     = jar.mean()
        mean_non_jar = non_jar.mean() if not non_jar.empty else np.nan
        penalty      = mean_jar - mean_non_jar

        std_jar = jar.std()
        md_low  = mean_jar - too_low.mean()  if len(too_low)  > 0 else np.nan
        md_high = mean_jar - too_high.mean() if len(too_high) > 0 else np.nan
        smd_low  = md_low  / std_jar if std_jar > 0 else np.nan
        smd_high = md_high / std_jar if std_jar > 0 else np.nan

        p_low  = mannwhitneyu(jar, too_low,  alternative='two-sided').pvalue if len(too_low)  > 0 else np.nan
        p_high = mannwhitneyu(jar, too_high, alternative='two-sided').pvalue if len(too_high) > 0 else np.nan
        _, p_pen = ttest_ind(jar, non_jar, nan_policy='omit') if not jar.empty and not non_jar.empty else (None, np.nan)

        results.append({
            'Attribute': abbreviation_dict.get(jar_col, jar_col),
            'Mean JAR': mean_jar,
            'Mean (Too Low)': too_low.mean(),
            'Pct Too Low':  len(too_low)  / len(df) * 100,
            'Mean Drop (Too Low)': md_low,
            'Std Mean Drop (Too Low)': smd_low,
            'p-value (Too Low)': p_low,
            'Mean (Too High)': too_high.mean(),
            'Pct Too High': len(too_high) / len(df) * 100,
            'Mean Drop (Too High)': md_high,
            'Std Mean Drop (Too High)': smd_high,
            'p-value (Too High)': p_high,
            'Penalty': penalty,
            'P-Value for Penalty': p_pen,
        })

    results_df = pd.DataFrame(results)
    results_df.to_excel(f'{path}{pa_save_name}.xlsx', index=False)

    # Chart 1: Scatter — Mean Drop vs % Respondents
    plt.figure(figsize=(10, 6))
    plt.axvline(x=threshold_pct, color='gray', linestyle='--', linewidth=1.5)
    for _, row in results_df.iterrows():
        attr = row['Attribute']
        plt.scatter(row['Pct Too Low'], row['Mean Drop (Too Low)'], color='cornflowerblue')
        plt.text(row['Pct Too Low'], row['Mean Drop (Too Low)'], attr, fontsize=9, ha='right')
        plt.scatter(row['Pct Too High'], row['Mean Drop (Too High)'], color='lightcoral')
        plt.text(row['Pct Too High'], row['Mean Drop (Too High)'], attr, fontsize=9, ha='left')
    plt.title(f'{pa_save_name} — Mean Drop vs % Respondents')
    plt.xlabel('% Respondents')
    plt.ylabel('Mean Drop in Liking')
    plt.axhline(0, color='black', linewidth=0.5)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'{path}PA_scatter_{pa_save_name}.png')
    plt.show()

    # Chart 2: Bar — Mean Drop per attribute
    mean_drops = results_df[['Attribute', 'Mean Drop (Too Low)', 'Mean Drop (Too High)',
                              'p-value (Too Low)', 'p-value (Too High)']].set_index('Attribute')
    fig, ax = plt.subplots(figsize=(10, 6))
    mean_drops[['Mean Drop (Too Low)', 'Mean Drop (Too High)']].plot(
        kind='bar', ax=ax, color=['cornflowerblue', 'lightcoral'])
    for bar, p in zip(ax.patches[:len(mean_drops)], mean_drops['p-value (Too Low)']):
        if p < 0.05: bar.set_hatch('/')
    for bar, p in zip(ax.patches[len(mean_drops):], mean_drops['p-value (Too High)']):
        if p < 0.05: bar.set_hatch('\\')
    plt.title(f'{pa_save_name} — Mean Drop')
    plt.xticks(rotation=45)
    plt.axhline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(f'{path}PA_bar_{pa_save_name}.png')
    plt.show()

    print(f"Penalty analysis saved to {path}{pa_save_name}.xlsx")

In [ ]:
# ── JAR Visualisation ─────────────────────────────────────────────────────────

def plot_likert_distribution(df, column_list, save_name):
    """
    Plot a stacked bar chart showing the percentage distribution
    of responses across all 5 Likert levels for each JAR attribute.
    Saves the chart and the underlying table to Excel.

    Parameters:
    - column_list (list): JAR columns to plot.
    - save_name (str): File name base for outputs.
    """
    df = df[column_list].copy()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    frequency_df   = df.apply(pd.value_counts).fillna(0)
    percentage_df  = frequency_df.div(frequency_df.sum()).multiply(100)
    percentage_df.to_excel(f'{path}{save_name}_Jar_5Pt_table.xlsx', index=True)

    fig, ax = plt.subplots(figsize=(16, 12))
    percentage_df.T.plot(kind='bar', stacked=True, colormap='viridis', ax=ax)
    wrapped = ['\n'.join(textwrap.wrap(lbl, 20)) for lbl in percentage_df.columns]
    ax.set_xticklabels(wrapped, rotation=0, ha='center')
    ax.set_title(f'{save_name} — Likert Distribution')
    ax.set_ylabel('Percentage (%)')
    plt.tight_layout()
    plt.savefig(f'{path}JAR_5Pt_{save_name}.png')
    plt.close()


def summarize_jar_data(df, jar_columns, abbreviation_dict, save_name):
    """
    Summarise JAR responses into Too Little / JAR / Too Much percentages.
    Plots a 3-category stacked bar chart and saves the table to Excel.

    Parameters:
    - jar_columns (list): JAR attribute columns.
    - abbreviation_dict (dict): {full_col_name: abbreviated_label}.
    - save_name (str): File name base for outputs.
    """
    df = df[jar_columns].copy()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    categories = {'Too Little': [1, 2], 'JAR': [3], 'Too Much': [4, 5]}
    summary    = {abbreviation_dict.get(c, c): {cat: 0 for cat in categories} for c in jar_columns}
    for long_name in jar_columns:
        short = abbreviation_dict.get(long_name, long_name)
        for cat, points in categories.items():
            summary[short][cat] = df[long_name].apply(lambda x: x in points).mean() * 100

    summary_df = pd.DataFrame(summary)
    summary_df.to_excel(f'{path}{save_name}_Jar_3Pt_table.xlsx', index=True)

    fig, ax = plt.subplots(figsize=(16, 12))
    summary_df.T.plot(kind='bar', stacked=True, colormap='viridis', ax=ax)
    wrapped = ['\n'.join(textwrap.wrap(lbl, 20)) for lbl in summary_df.columns]
    ax.set_xticklabels(wrapped, rotation=0, ha='center')
    plt.title(f'{save_name} — JAR 3-Category Distribution')
    plt.tight_layout()
    plt.savefig(f'{path}JAR_3Pt_{save_name}.png')
    plt.close()

In [ ]:
# ── CATA Analysis ─────────────────────────────────────────────────────────────
# Check-All-That-Apply analysis counts attribute selection frequency
# and tests for significant differences between products using McNemar's test.

def cata_sheet_name(s_name):
    """Extract a valid 31-character Excel sheet name from a question string."""
    match = re.search(r'(.{31})\?', s_name)
    return match.group(1) if match else s_name[-31:-1]


def cata_analysis_to_excel(df, cata_columns, output_file):
    """
    Count CATA attribute selections and save frequency tables to Excel.
    One sheet per CATA question.

    Parameters:
    - cata_columns (list): Columns containing comma-separated CATA responses.
    - output_file (str): Output Excel file path.
    """
    writer = pd.ExcelWriter(output_file, engine='xlsxwriter')
    for col in cata_columns:
        split_series = df[col].dropna().str.split(',').explode()
        counts       = split_series.value_counts()
        percentages  = (counts / len(df[col].dropna())) * 100
        result = pd.DataFrame({
            'Attribute': counts.index,
            'Count': counts.values,
            'Percentage': percentages.values
        }).reset_index(drop=True)
        result.to_excel(writer, sheet_name=cata_sheet_name(col), index=False)
    writer.close()
    print(f"CATA analysis saved to {output_file}")


def preprocess_cata_column(df, column):
    """
    Convert a comma-separated CATA column into binary indicator columns.
    Each unique attribute becomes a column with 0/1 values.

    Returns:
    - pd.DataFrame of binary columns prefixed with the column name.
    """
    binary_df = df[column].str.get_dummies(sep=',')
    return binary_df.add_prefix(f"{column}_")


def compare_cata_mcnemar(df1, df2, cata_cols):
    """
    Perform McNemar's test for each CATA attribute across two DataFrames.

    Parameters:
    - df1, df2 (pd.DataFrame): DataFrames for each product.
    - cata_cols (list): CATA question columns.

    Returns:
    - pd.DataFrame: p-values indexed by (Column, Attribute).
    """
    all_results = []
    for col in cata_cols:
        if col not in df1.columns or col not in df2.columns:
            raise ValueError(f"Column '{col}' not found in both DataFrames.")
        b1 = preprocess_cata_column(df1, col)
        b2 = preprocess_cata_column(df2, col)
        common = list(set(b1.columns) & set(b2.columns))
        if not common:
            continue
        for attr in common:
            table = pd.crosstab(b1[attr], b2[attr])
            if table.shape != (2, 2):
                continue
            result    = mcnemar(table, exact=True)
            short_attr = attr.split('_')[-1]
            all_results.append(((col, short_attr), result.pvalue))

    results_df = pd.DataFrame(all_results, columns=['Column_Attribute', 'p-value'])
    results_df[['Column', 'Attribute']] = pd.DataFrame(
        results_df['Column_Attribute'].tolist(), index=results_df.index
    )
    return results_df.drop(columns=['Column_Attribute']).set_index(['Column', 'Attribute'])


def batch_compare_cata_mcnemar(dfs, df_names, cata_cols, path=""):
    """
    Compare all pairwise combinations of DataFrames for CATA attributes.
    Saves each comparison to a separate Excel file.

    Parameters:
    - dfs (list of pd.DataFrame): Product DataFrames.
    - df_names (list of str): Labels for each DataFrame.
    - cata_cols (list): CATA question columns.
    - path (str): Output folder.

    Returns:
    - dict: {(name1, name2): results_df}.
    """
    results = {}
    for (df1, name1), (df2, name2) in itertools.combinations(zip(dfs, df_names), 2):
        print(f"Comparing {name1} vs {name2}...")
        comparison = compare_cata_mcnemar(df1, df2, cata_cols)
        results[(name1, name2)] = comparison
        comparison.to_excel(f"{path}CATA_SigDiff_{name1}_vs_{name2}.xlsx")
    return results


def penalty_analysis(cata_column, liking_scores, question_name):
    """
    CATA Penalty Analysis: measures impact of each attribute on liking.
    Penalty = Mean Liking (attribute checked) − Mean Liking (not checked).
    Significant penalties indicate the attribute meaningfully influences liking.

    Parameters:
    - cata_column (pd.Series): Comma-separated CATA responses.
    - liking_scores (pd.Series): Overall liking scores aligned by index.
    - question_name (str): CATA question label for output naming.

    Returns:
    - pd.DataFrame: Results with attribute, means, penalty, and p-value.
    """
    cata_series = cata_column.fillna('').astype(str)
    liking      = pd.to_numeric(liking_scores, errors='coerce')
    token_lists = cata_series.str.split(',').apply(lambda xs: [x.strip() for x in xs if x.strip()])
    unique_attrs = sorted({tok for row in token_lists for tok in row})

    if not unique_attrs:
        raise ValueError(f"No CATA attributes found for: {question_name}")

    binary_matrix = pd.DataFrame(
        {attr: cata_series.str.contains(re.escape(attr), na=False).astype(int)
         for attr in unique_attrs},
        index=cata_series.index
    )
    binary_matrix['Liking'] = liking

    results = []
    for attr in unique_attrs:
        checked     = binary_matrix.loc[binary_matrix[attr] == 1, 'Liking'].dropna()
        not_checked = binary_matrix.loc[binary_matrix[attr] == 0, 'Liking'].dropna()
        mean_c  = checked.mean()
        mean_nc = not_checked.mean()
        penalty = mean_c - mean_nc
        p_value = ttest_ind(checked, not_checked, equal_var=False).pvalue if len(checked) > 1 and len(not_checked) > 1 else np.nan
        results.append((attr, mean_c, mean_nc, penalty, p_value))

    results_df = pd.DataFrame(results, columns=['Attribute', 'Mean_Checked', 'Mean_Not_Checked', 'Penalty', 'P_Value'])
    q_sheet    = cata_sheet_name(question_name)
    results_df.to_excel(f'{path}{q_sheet}.xlsx', index=False)

    sig = results_df[results_df['P_Value'] < 0.05]
    plt.figure(figsize=(16, 12))
    plt.barh(sig['Attribute'], sig['Penalty'])
    plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
    plt.xlabel('Penalty Value')
    plt.title(f'CATA Penalty Analysis — {q_sheet}')
    plt.savefig(f'{path}CATA_PA_{q_sheet}.png')
    plt.show()

    return results_df

In [ ]:
# ── MaxDiff Analysis ──────────────────────────────────────────────────────────

def maxdiff_analysis(df, max_diff_columns, save_name):
    """
    Analyse MaxDiff survey data and generate bar and quadrant charts.
    MaxDiff responses are formatted as 'Attribute:M' or 'Attribute:L'
    within each column, comma-separated.

    Net Score = Most Important count − Least Important count.

    Parameters:
    - max_diff_columns (list): Columns containing MaxDiff responses.
    - save_name (str): File name base for output charts.

    Returns:
    - pd.DataFrame: Summary with Most, Least, and Net Score per attribute.
    """
    most_counter  = Counter()
    least_counter = Counter()

    for col in max_diff_columns:
        for responses in df[col].dropna():
            for response in responses.split(','):
                attribute, choice = response.rsplit(':', 1)
                attribute = attribute.strip()
                if choice.strip() == 'M': most_counter[attribute]  += 1
                elif choice.strip() == 'L': least_counter[attribute] += 1

    all_attrs  = set(most_counter) | set(least_counter)
    summary_df = pd.DataFrame({
        'Attribute': list(all_attrs),
        'Most Important (M)': [most_counter[a]  for a in all_attrs],
        'Least Important (L)': [least_counter[a] for a in all_attrs]
    })
    summary_df['Net Score'] = summary_df['Most Important (M)'] - summary_df['Least Important (L)']
    summary_df.sort_values('Net Score', ascending=False, inplace=True)

    # Bar chart
    plt.figure(figsize=(10, 6))
    idx = range(len(summary_df))
    plt.bar(idx, summary_df['Most Important (M)'], 0.4, label='Most Important')
    plt.bar(idx, -summary_df['Least Important (L)'], 0.4, color='salmon', label='Least Important')
    plt.axhline(0, color='black', linewidth=0.8)
    plt.xticks(idx, summary_df['Attribute'], rotation=45, ha='right')
    plt.title(f'{save_name} — MaxDiff')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{path}{save_name}_bar_chart.png')
    plt.close()

    # Quadrant chart
    plt.figure(figsize=(8, 8))
    plt.scatter(summary_df['Most Important (M)'], summary_df['Least Important (L)'], s=100, alpha=0.7)
    for _, row in summary_df.iterrows():
        plt.text(row['Most Important (M)'], row['Least Important (L)'],
                 row['Attribute'], fontsize=9, ha='right')
    max_most  = summary_df['Most Important (M)'].max()
    max_least = summary_df['Least Important (L)'].max()
    plt.axhline(max_least / 2, color='gray', linestyle='--', linewidth=0.8)
    plt.axvline(max_most  / 2, color='gray', linestyle='--', linewidth=0.8)
    plt.title(f'{save_name} — MaxDiff Quadrant')
    plt.tight_layout()
    plt.savefig(f'{path}{save_name}_quadrant_chart.png')
    plt.close()

    print(f"MaxDiff charts saved to {path}")
    return summary_df

---
## SECTION 6 — Driver Analysis
Five methods for identifying which product attributes drive overall liking or retrial intent. All methods handle missing data via a configurable threshold. Results are merged into a single comparison table.

In [ ]:
# ── Driver Analysis Methods ───────────────────────────────────────────────────
# All driver functions share the same interface:
#   df, predictors, target, missing_threshold, verbose → results DataFrame
#
# Missing threshold: columns with more than this proportion of NaN are excluded.
# All methods normalise output to relative weights summing to 100%.
#
# Method selection guidance:
#   LR         — baseline, interpretable, sensitive to multicollinearity
#   Johnson    — corrects for multicollinearity via PCA orthogonalisation
#   RF         — non-parametric, captures non-linear effects
#   HGB        — handles high missingness natively, best for sparse data
#   Lasso      — feature selection, zeroes out weak predictors
#   OLS Signed — preserves coefficient sign (positive/negative relationship)

def compute_linear_weights_LR_v2(df, predictors, target, missing_threshold=0.3, verbose=True):
    """
    Compute feature importances using standardised Linear Regression coefficients.
    Absolute coefficients are normalised to relative weights (% summing to 100).

    Parameters:
    - df (pd.DataFrame): Dataset.
    - predictors (list): Predictor column names.
    - target (str): Target column name (Overall Liking / Retrial Intent).
    - missing_threshold (float): Max allowed missingness proportion per column.
    - verbose (bool): Print dropped columns and row counts.

    Returns:
    - pd.DataFrame: ['Variable', 'Linear Relative_Weight (%)'] sorted descending.
    """
    if target in predictors:
        predictors = [c for c in predictors if c != target]
    valid = [c for c in predictors if df[c].isnull().mean() <= missing_threshold]
    if verbose and set(predictors) - set(valid):
        print(f"  LR dropped: {list(set(predictors) - set(valid))}")
    data = df[valid + [target]].dropna()
    if verbose: print(f"  LR rows: {len(data)}")
    if data.empty:
        return pd.DataFrame(columns=['Variable', 'Linear Relative_Weight (%)'])
    X = StandardScaler().fit_transform(data[valid])
    y = StandardScaler().fit_transform(data[target].values.reshape(-1, 1)).ravel()
    model = LinearRegression().fit(X, y)
    coefs = np.abs(model.coef_)
    weights = 100 * coefs / np.sum(coefs) if np.sum(coefs) > 0 else np.zeros_like(coefs)
    return pd.DataFrame({'Variable': valid, 'Linear Relative_Weight (%)': weights}).sort_values(
        'Linear Relative_Weight (%)', ascending=False).reset_index(drop=True)


def compute_johnson_weights_v2(df, predictors, target, missing_threshold, verbose=True):
    """
    Compute Johnson's Relative Weights via PCA orthogonalisation.
    Decomposes R² into contributions from each predictor, correcting
    for multicollinearity. Preferred method when Condition Index is high.

    Returns:
    - pd.DataFrame: ['Variable', 'Johnson_Relative_Weight (%)'] sorted descending.
    """
    if target in predictors:
        predictors = [c for c in predictors if c != target]
    valid = [c for c in predictors if df[c].isnull().mean() <= missing_threshold]
    if verbose and set(predictors) - set(valid):
        print(f"  Johnson dropped: {list(set(predictors) - set(valid))}")
    data = df[valid + [target]].dropna()
    if verbose: print(f"  Johnson rows: {len(data)}")
    if data.empty:
        return pd.DataFrame(columns=['Variable', 'Johnson_Relative_Weight (%)'])
    X_scaled = StandardScaler().fit_transform(data[valid])
    y        = data[target].values
    pca      = PCA().fit(X_scaled)
    Z        = pca.transform(X_scaled)
    beta_z   = LinearRegression().fit(Z, y).coef_
    struct   = np.corrcoef(X_scaled.T, Z.T)[:len(valid), len(valid):]
    raw_w    = np.dot(struct ** 2, beta_z ** 2)
    rel_w    = 100 * raw_w / np.sum(raw_w)
    return pd.DataFrame({'Variable': valid, 'Johnson_Relative_Weight (%)': rel_w}).sort_values(
        'Johnson_Relative_Weight (%)', ascending=False).reset_index(drop=True)


def compute_random_forest_importance_v2(df, predictors, target, missing_threshold, verbose=True):
    """
    Compute Random Forest feature importances (Gini impurity based).
    Captures non-linear and interaction effects. No scaling required.

    Returns:
    - pd.DataFrame: ['Variable', 'RF Relative_Weight (%)'] sorted descending.
    """
    if target in predictors:
        predictors = [c for c in predictors if c != target]
    valid = [c for c in predictors if df[c].isnull().mean() <= missing_threshold]
    if verbose and set(predictors) - set(valid):
        print(f"  RF dropped: {list(set(predictors) - set(valid))}")
    data = df[valid + [target]].dropna()
    if verbose: print(f"  RF rows: {len(data)}")
    if data.empty:
        return pd.DataFrame(columns=['Variable', 'RF Relative_Weight (%)'])
    model = RandomForestRegressor(n_estimators=100, random_state=42,
                                   max_features='sqrt', n_jobs=-1)
    model.fit(data[valid], data[target])
    raw = model.feature_importances_
    rel = 100 * raw / np.sum(raw)
    return pd.DataFrame({'Variable': valid, 'RF Relative_Weight (%)': rel}).sort_values(
        'RF Relative_Weight (%)', ascending=False).reset_index(drop=True)


def compute_hgb_importance_v1(df, predictors, target, missing_threshold, verbose=True):
    """
    Compute feature importances using HistGradientBoosting with permutation importance.
    Handles missing values natively — preferred for sparse datasets (>30% missing).
    Binary (0/1) columns are zero-filled for absent attributes.

    Returns:
    - pd.DataFrame: ['Variable', 'HGB Permutation_Weight (%)'] sorted descending.
    """
    mt = missing_threshold / 100.0 if missing_threshold > 1 else float(missing_threshold)
    if target in predictors:
        predictors = [c for c in predictors if c != target]
    miss_rates = df[predictors].isna().mean()
    valid = [c for c in predictors if miss_rates.get(c, 1.0) <= mt]
    if verbose and set(predictors) - set(valid):
        print(f"  HGB dropped: {sorted(set(predictors) - set(valid))}")
    if not valid:
        return pd.DataFrame(columns=['Variable', 'HGB Permutation_Weight (%)'])
    X = df[valid].copy()
    y = df[target]
    ohe_cols = [c for c in valid if X[c].dropna().isin([0, 1]).all()]
    if ohe_cols:
        X[ohe_cols] = X[ohe_cols].fillna(0)
    mask = y.notna()
    X, y = X.loc[mask], y.loc[mask]
    if verbose: print(f"  HGB rows: {len(y)}")
    if X.empty:
        return pd.DataFrame(columns=['Variable', 'HGB Permutation_Weight (%)'])
    model = HistGradientBoostingRegressor(
        learning_rate=0.1, max_leaf_nodes=32, min_samples_leaf=20,
        early_stopping=True, n_iter_no_change=10, random_state=42
    )
    model.fit(X, y)
    pi  = permutation_importance(model, X, y, n_repeats=5, random_state=42,
                                  n_jobs=-1, scoring='neg_mean_squared_error')
    imp = np.clip(pi.importances_mean, 0, None)
    total = imp.sum()
    rel   = 100.0 * imp / total if total > 0 else np.zeros_like(imp)
    return pd.DataFrame({'Variable': valid, 'HGB Permutation_Weight (%)': rel}).sort_values(
        'HGB Permutation_Weight (%)', ascending=False).reset_index(drop=True)


def compute_lasso_importance_v2(df, predictors, target, missing_threshold=0.3, verbose=True):
    """
    Compute feature importances using Lasso regression (L1 regularisation).
    Lasso applies feature selection by zeroing weak predictors.
    Useful for identifying the most parsimonious set of drivers.

    Returns:
    - pd.DataFrame: ['Variable', 'Lasso Relative_Weight (%)'] sorted descending.
    """
    if target in predictors:
        predictors = [c for c in predictors if c != target]
    valid = [c for c in predictors if df[c].isnull().mean() <= missing_threshold]
    if verbose and set(predictors) - set(valid):
        print(f"  Lasso dropped: {list(set(predictors) - set(valid))}")
    data = df[valid + [target]].dropna()
    if verbose: print(f"  Lasso rows: {len(data)}")
    if data.empty:
        return pd.DataFrame(columns=['Variable', 'Lasso Relative_Weight (%)'])
    X = StandardScaler().fit_transform(data[valid])
    y = StandardScaler().fit_transform(data[target].values.reshape(-1, 1)).ravel()
    model = Lasso(alpha=0.01, random_state=42, max_iter=10000).fit(X, y)
    coefs = np.abs(model.coef_)
    rel   = 100 * coefs / np.sum(coefs) if np.sum(coefs) > 0 else np.zeros_like(coefs)
    return pd.DataFrame({'Variable': valid, 'Lasso Relative_Weight (%)': rel}).sort_values(
        'Lasso Relative_Weight (%)', ascending=False).reset_index(drop=True)


def ols_signed_coefficients(df, predictors, target, standardize=True, verbose=True):
    """
    Fit OLS and return signed standardised coefficients.
    Unlike other driver methods, this preserves the direction of the relationship
    (positive = attribute drives liking up; negative = attribute drives liking down).
    Useful for understanding whether attributes are positively or negatively driven.

    Returns:
    - pd.DataFrame: ['Variable', 'OLS Coefficient'] sorted descending.
    """
    seen = set()
    predictors = [p for p in predictors if not (p in seen or seen.add(p))]
    cols = predictors + [target]
    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Columns not found: {missing_cols}")
    data = df[cols].apply(pd.to_numeric, errors='coerce').dropna()
    if verbose: print(f"  OLS rows: {len(data)}")
    if data.shape[0] < 2:
        raise ValueError("Not enough rows after dropna.")
    stds  = data[predictors].std()
    keep  = list(stds[stds > 0].index)
    if not keep:
        raise ValueError("All predictors have zero variance.")
    X = data[keep].to_numpy()
    y = data[target].to_numpy()
    if standardize:
        X = StandardScaler().fit_transform(X)
        y = StandardScaler().fit_transform(y.reshape(-1, 1)).ravel()
    model  = sm.OLS(y, sm.add_constant(X, has_constant='add')).fit()
    coefs  = model.params[1:]
    if verbose:
        print(f"  OLS R²={model.rsquared:.4f}  adj_R²={model.rsquared_adj:.4f}")
    return pd.DataFrame({'Variable': keep, 'OLS Coefficient': coefs}).sort_values(
        'OLS Coefficient', ascending=False).reset_index(drop=True)


def johnson_analysis(df, columns_10pt, columns_5pt, target_col, save_name):
    """
    Run Johnson's Relative Weights and save to Excel.
    Convenience wrapper around compute_johnson_weights_v2.

    Returns:
    - tuple (johnson_df, file_path).
    """
    predictors  = columns_10pt + columns_5pt
    johnson_df  = compute_johnson_weights_v2(df, predictors, target_col, missing_threshold=0.3)
    file_path   = f"{path}JWS_{save_name}.xlsx"
    johnson_df.to_excel(file_path, index=False)
    return johnson_df, os.path.abspath(file_path)

In [ ]:
# ── Driver Analysis Pipeline ──────────────────────────────────────────────────
# Runs all five driver methods and merges results into a single comparison table.
# Allows cross-method validation — consistent rankings across methods indicate
# robust drivers; disagreement indicates potential multicollinearity or non-linearity.

def run_driver_analysis(df, predictors, target_column, missing_threshold,
                         save_name, run_hgb=True):
    """
    Run all driver analysis methods and save merged results to Excel.

    Parameters:
    - df (pd.DataFrame): Dataset.
    - predictors (list): Predictor columns.
    - target_column (str): Overall liking or retrial intent column.
    - missing_threshold (float): Max allowed missingness.
    - save_name (str): Filename base for output.
    - run_hgb (bool): Include HGB (slower, run only when needed).

    Returns:
    - pd.DataFrame: Merged driver analysis results.
    """
    print(f"Running driver analysis for: {save_name}")

    lr_df       = compute_linear_weights_LR_v2(df, predictors, target_column, missing_threshold)
    johnson_df  = compute_johnson_weights_v2(df, predictors, target_column, missing_threshold)
    rf_df       = compute_random_forest_importance_v2(df, predictors, target_column, missing_threshold)
    lasso_df    = compute_lasso_importance_v2(df, predictors, target_column, missing_threshold)

    merged = lr_df.merge(johnson_df, on='Variable', how='outer')
    merged = merged.merge(rf_df,     on='Variable', how='outer')
    merged = merged.merge(lasso_df,  on='Variable', how='outer')

    if run_hgb:
        hgb_df = compute_hgb_importance_v1(df, predictors, target_column, missing_threshold)
        merged = merged.merge(hgb_df, on='Variable', how='outer')

    output_path = f"{path}_DA_Combined_{save_name}.xlsx"
    merged.to_excel(output_path, index=False)
    print(f"Driver analysis saved to {output_path}")
    return merged

---
## SECTION 7 — Translation & NLP
Open-ended response translation and thematic analysis using TF-IDF and NMF topic modelling.

In [ ]:
# ── Translation ───────────────────────────────────────────────────────────────
# Two translation implementations:
#   translate_columns      — uses deep-translator (free, rate-limited)
#   translate_columns_google_api — uses Google Cloud Translation API (key required)

def translate_columns(df, cols, output_file, source_lang, target_lang):
    """
    Translate specified columns using deep-translator (GoogleTranslator).
    Only translates cells where the detected language matches source_lang.
    Saves the translated DataFrame to Excel.

    Parameters:
    - cols (list): Columns to translate.
    - source_lang (str): Source language code (e.g. 'nl', 'fr', 'de').
    - target_lang (str): Target language code (e.g. 'en').

    Returns:
    - pd.DataFrame with translated columns.
    """
    translator = GoogleTranslator(source=source_lang, target=target_lang)

    def translate_if_needed(text):
        try:
            if detect(text) == source_lang:
                return translator.translate(text)
            return text
        except:
            return text

    for col in cols:
        if col in df.columns:
            print(f"  Translating: {col}")
            df[col] = df[col].fillna('').astype(str).apply(
                lambda x: translate_if_needed(x) if x.strip() else ''
            )
        else:
            print(f"  Warning: '{col}' not found.")

    df.to_excel(output_file, index=False)
    print(f"Translated data saved to {output_file}")
    return df


def translate_columns_google_api(df, cols, output_file, source_lang, target_lang, api_key=None):
    """
    Translate specified columns using Google Cloud Translation API.
    Requires a valid API key set via the GOOGLE_TRANSLATE_API_KEY environment variable.
    More reliable than deep-translator for production use — handles special characters
    and longer texts correctly.

    Parameters:
    - cols (list): Columns to translate.
    - source_lang (str): Source language code.
    - target_lang (str): Target language code.
    - api_key (str): Google Cloud API key. Defaults to environment variable.

    Returns:
    - pd.DataFrame with translated columns.
    """
    api_key = api_key or GOOGLE_TRANSLATE_API_KEY

    def translate_text(text):
        if not isinstance(text, str) or text.strip().lower() in {'', 'nan', 'none'}:
            return text
        url    = 'https://translation.googleapis.com/language/translate/v2'
        params = {'q': text, 'source': source_lang, 'target': target_lang,
                  'format': 'text', 'key': api_key}
        try:
            response = requests.post(url, data=params)
            response.raise_for_status()
            return response.json()['data']['translations'][0]['translatedText']
        except requests.exceptions.RequestException as e:
            print(f"  Translation failed for '{text[:40]}...': {e}")
            return text

    for col in cols:
        if col in df.columns:
            print(f"  Translating: {col}")
            df[col] = df[col].astype(str).apply(translate_text)
        else:
            print(f"  Warning: '{col}' not found.")

    df.to_excel(output_file, index=False)
    print(f"Translated data saved to {output_file}")
    return df

In [ ]:
# ── Thematic Analysis ─────────────────────────────────────────────────────────
# Automated theme discovery from open-ended survey responses using
# TF-IDF vectorisation and Non-negative Matrix Factorisation (NMF).
#
# Outputs:
#   - Excel workbook with Overview, per-column Summary, Example Quotes, and Verbatims sheets
#   - JSON dictionary of theme labels and top terms (reusable for future waves)

def perform_thematic_analysis_english(
    df, oe_columns, save_name, out_dir='.',
    n_topics=None, max_features=5000, top_terms=8,
    exemplar_quotes_per_theme=3, min_df=2,
    assign_threshold=0.35, random_state=42
):
    """
    Discover themes from English open-ended survey responses.
    Topic count is auto-estimated from corpus size if n_topics is None.

    Parameters:
    - df (pd.DataFrame): Dataset containing open-ended columns.
    - oe_columns (list): Open-ended text columns to analyse.
    - save_name (str): Filename base for outputs.
    - out_dir (str): Output directory.
    - n_topics (int or None): Number of topics. Auto-estimated if None.
    - max_features (int): TF-IDF vocabulary size limit.
    - top_terms (int): Number of top terms per theme.
    - exemplar_quotes_per_theme (int): Example quotes per theme in output.
    - min_df (int): Minimum document frequency for TF-IDF terms.
    - assign_threshold (float): Fraction of max topic score to assign a theme.
    - random_state (int): NMF random seed.

    Returns:
    - dict: {'excel_path', 'dictionary_path', 'columns_processed'}.
    """
    _used_sheets = set()

    def _safe_sheet(base, suffix=''):
        clean = re.sub(r'[:\\/?*\[\]]', ' ', str(base)).strip()
        maxl  = 31 - len(suffix)
        name  = clean[:max(maxl, 1)].rstrip() + suffix
        k = 2
        while name in _used_sheets:
            tail = f"_{k}"
            name = name[:31 - len(tail)] + tail
            k   += 1
        _used_sheets.add(name)
        return name

    os.makedirs(out_dir, exist_ok=True)
    excel_path = os.path.join(out_dir, f"{save_name}_Thematic_Analysis.xlsx")
    dict_path  = os.path.join(path, f"{save_name}_themes.json")

    def clean_text(s):
        s = str(s).replace('\u200b', ' ')
        return re.sub(r'\s+', ' ', s).strip()

    def auto_topics(n):
        return max(3, min(8, int(round(2 * math.log(n))))) if n > 10 else 3

    overview_rows, verbatims_rows = [], []
    percol_summaries, percol_examples = {}, {}
    dictionary = {
        'save_name': save_name, 'language': 'en', 'columns': {},
        'meta': {'method': 'TF-IDF + NMF', 'assign_threshold': assign_threshold,
                 'min_df': min_df, 'max_features': max_features,
                 'top_terms': top_terms, 'random_state': random_state}
    }

    for col in oe_columns:
        if col not in df.columns: continue
        series = df[col].dropna()
        if series.empty: continue
        texts  = series.astype(str).map(clean_text)
        texts  = texts[texts.str.len() > 0]
        doc_idx = texts.index.tolist()
        if texts.empty:
            percol_summaries[col] = pd.DataFrame(columns=['Theme','Count','% of Responses','Top Terms'])
            percol_examples[col]  = {}
            dictionary['columns'][col] = {'themes': []}
            continue

        k = n_topics or auto_topics(len(texts))
        tfidf = TfidfVectorizer(lowercase=True, stop_words='english',
                                 max_features=max_features, min_df=min_df, ngram_range=(1, 2))
        try:
            X = tfidf.fit_transform(texts)
        except ValueError:
            tfidf = TfidfVectorizer(lowercase=True, stop_words=None, max_features=max_features,
                                     min_df=1, max_df=1.0, ngram_range=(1, 1))
            X = tfidf.fit_transform(texts)

        if X.shape[1] == 0:
            percol_summaries[col] = pd.DataFrame(columns=['Theme','Count','% of Responses','Top Terms'])
            percol_examples[col]  = {}
            continue

        k   = min(k, X.shape[0], X.shape[1])
        nmf = NMF(n_components=k, random_state=random_state, init='nndsvda', max_iter=400)
        W   = nmf.fit_transform(X)
        H   = nmf.components_
        vocab = np.array(tfidf.get_feature_names_out())

        theme_terms = [vocab[H[t].argsort()[::-1][:top_terms]].tolist() for t in range(k)]
        theme_labels = [f"Theme {t+1}: " + ', '.join(theme_terms[t][:3]) for t in range(k)]

        max_per_doc = W.max(axis=1)
        max_per_doc[max_per_doc == 0] = 1e-9
        assignments = [np.where(W[i] >= assign_threshold * max_per_doc[i])[0].tolist() or [int(np.argmax(W[i]))]
                       for i in range(W.shape[0])]

        theme_counts = np.zeros(k, dtype=int)
        exemplars    = defaultdict(list)
        for i, t_ids in enumerate(assignments):
            for t in t_ids:
                theme_counts[t] += 1
                if len(exemplars[t]) < exemplar_quotes_per_theme * 5:
                    exemplars[t].append((W[i, t], texts.iloc[i]))

        n_resp    = len(texts)
        col_summary = pd.DataFrame({
            'Theme': theme_labels,
            'Count': theme_counts,
            '% of Responses': (theme_counts / n_resp * 100).round(1),
            'Top Terms': [', '.join(theme_terms[t]) for t in range(k)]
        }).sort_values('Count', ascending=False).reset_index(drop=True)
        percol_summaries[col] = col_summary

        percol_examples[col] = {
            theme_labels[t]: [q for _, q in sorted(exemplars[t], key=lambda x: x[0], reverse=True)[:exemplar_quotes_per_theme]]
            for t in range(k)
        }
        dictionary['columns'][col] = {'themes': [{'label': theme_labels[t], 'top_terms': theme_terms[t]} for t in range(k)]}

        for i_local, idx in enumerate(doc_idx):
            verbatims_rows.append((col, idx, df.at[idx, col], ' | '.join(theme_labels[t] for t in assignments[i_local])))
        for t, label in enumerate(theme_labels):
            overview_rows.append({'OE Column': col, 'Theme': label, 'Count': int(theme_counts[t]),
                                   '% of Responses': float(theme_counts[t] / n_resp * 100),
                                   'Top Terms': ', '.join(theme_terms[t])})

    overview_df   = pd.DataFrame(overview_rows)
    verbatims_df  = pd.DataFrame(verbatims_rows, columns=['OE Column', 'Row Index', 'Verbatim', 'Assigned Themes'])

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        if not overview_df.empty:
            overview_df.sort_values(['OE Column', 'Count'], ascending=[True, False]).reset_index(drop=True).to_excel(
                writer, sheet_name='Overview', index=False)
        for col, summary in percol_summaries.items():
            summary.to_excel(writer, sheet_name=_safe_sheet(col, '_Summary'), index=False)
            rows = [{'Theme': t, 'Exemplar Quote': q}
                    for t, quotes in percol_examples.get(col, {}).items() for q in (quotes or [''])]
            pd.DataFrame(rows).to_excel(writer, sheet_name=_safe_sheet(col, '_Examples'), index=False)
        if not verbatims_df.empty:
            verbatims_df.to_excel(writer, sheet_name='Verbatims', index=False)

    with open(dict_path, 'w', encoding='utf-8') as f:
        json.dump(dictionary, f, ensure_ascii=False, indent=2)

    print(f"Thematic analysis saved to {excel_path}")
    return {'excel_path': excel_path, 'dictionary_path': dict_path,
            'columns_processed': [c for c in oe_columns if c in df.columns]}

In [ ]:
# ── Logging Setup ─────────────────────────────────────────────────────────────

def setup_logging(logfile_path):
    """
    Redirect all stdout and stderr to a log file.
    Useful for long overnight batch runs.

    Parameters:
    - logfile_path (str): Path to the log file.
    """
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    file_handler    = logging.FileHandler(logfile_path, mode='w')
    console_handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    class StreamToLogger:
        def __init__(self, level):
            self.level = level
        def write(self, message):
            if message.strip(): self.level(message.strip())
        def flush(self):
            pass

    sys.stdout = StreamToLogger(logger.info)
    sys.stderr = StreamToLogger(logger.error)

---
## SECTION 8 — Run Pipeline
Orchestration cell. Load your study DataFrames from HDF5, configure the analysis parameters, and run the full pipeline.

In [ ]:
# ── Pipeline Configuration ────────────────────────────────────────────────────
# Customise these for each study before running.

# List of HDF5 keys to process (one per product/rotation)
df_mapped_list = [
    # 'df_mapped_ProductA_D3',
    # 'df_mapped_ProductB_D3',
]

# Target column for driver analysis and penalty analysis
target_column = ""  # e.g. overall opinion or retrial intent question text
Liking        = target_column

# JAR scale direction
# True  = inverted scale (1–2 = Too Much, 3 = JAR, 4–5 = Too Little)
# False = standard scale (1–2 = Too Little, 3 = JAR, 4–5 = Too Much)
JAR_inverted = True

# Missing data threshold for driver analysis
missing_threshold = 0.30

# Abbreviated labels for JAR attributes (used in chart labels)
abbreviations = {}  # {full_question_text: short_label}

# Retrial intent questions to exclude from driver analysis
RI_Remove = []  # questions that are retrial intent (leakage from target)
RI_Remove_ALL = []  # broader exclusion list including low-frequency questions
RI_Remove_ALL = RI_Remove_ALL + RI_Remove

In [ ]:
# ── Run Full Pipeline ─────────────────────────────────────────────────────────
# Loads all product DataFrames from HDF5 and runs the full analysis sequence.
#
# Two predictor sets are used for driver analysis:
#
#   predictors (RI_Remove)
#     Removes only the primary OO/RI question from the predictor list.
#     Used for HGB — run with high missingness threshold (0.99) to capture
#     all attributes including sparse ones.
#
#   predictors_ALL (RI_Remove_ALL)
#     Removes OO, RI, and low-occurrence questions.
#     Used for LR, Johnson, RF, Lasso — cleaner predictor set for
#     methods sensitive to sparse columns.

df_mapped_list_of_df = []
with pd.HDFStore(h5_file_path, mode='r') as store:
    for key in df_mapped_list:
        df_mapped_list_of_df.append(store.get(key))

for df, save_name in zip(df_mapped_list_of_df, df_mapped_list):
    print(f"\n{'='*60}\nProcessing: {save_name}\n{'='*60}")

    # ── Normalise 10-point to 5-point for correlation ─────────────────────
    normal_df     = convert_10_to_5(df.copy(), all_10pt_cols)
    cor_save_name = f"{save_name}_corr"
    spearman_rank_correlation(normal_df, cor_save_name, T2B_TEN)

    # ── Multicollinearity check ───────────────────────────────────────────
    calculate_condition_index_and_variance_decomposition_v2(
        df, T2B_cols, all_10pt_cols, save_name, missing_threshold)

    # ── JAR Penalty Analysis ──────────────────────────────────────────────
    pa_save_name = f"{save_name}_PA"
    if JAR_inverted:
        penalty_analysis_invert_scale(
            df, jar_cols, target_column, abbreviations, pa_save_name=pa_save_name)
    else:
        # Standard (non-inverted) scale
        for col_name in jar_cols:
            penalty_analysis(df[col_name], df[Liking], col_name)

    # JAR distribution charts (5-point and 3-category)
    plot_likert_distribution(df, jar_cols, pa_save_name)
    summarize_jar_data(df, jar_cols, abbreviations, pa_save_name)

    # ── MaxDiff (uncomment if study includes MaxDiff) ─────────────────────
    # md_save_name = f"MD_{save_name}"
    # md_df = maxdiff_analysis(df, md_cols, md_save_name)

    # ── CATA Analysis ─────────────────────────────────────────────────────
    # Frequency counts
    cata_analysis_to_excel(df, CATA_cols, f"{path}CATA_{save_name}.xlsx")
    cata_analysis_to_excel(df, CATA_cols, f"{path}CATA_sum_CT_{save_name}.xlsx")

    # CATA Penalty Analysis — one run per CATA question
    for col_name in CATA_cols:
        print(f"  CATA PA: {col_name}")
        penalty_analysis(df[col_name], df[Liking], col_name)

    # ── Driver Analysis — Pass 1: HGB on broader predictor set ───────────
    # HGB handles missingness natively so a wider predictor set is viable.
    # Uses RI_Remove only (not RI_Remove_ALL) to keep low-occurrence Qs.
    RI_set           = set(RI_Remove)
    predictors_R_T2B = [c for c in T2B_cols      if c not in RI_set]
    predictors_R_TEN = [c for c in all_10pt_cols  if c not in RI_set]
    predictors       = predictors_R_T2B + predictors_R_TEN

    hgb_df = compute_hgb_importance_v1(
        df, predictors, target_column, missing_threshold=0.99)
    hgb_df.to_excel(f"{path}_DA_hgb_ONLY_{save_name}.xlsx", index=False)

    # ── Driver Analysis — Pass 2: LR, Johnson, RF, Lasso on clean set ────
    # RI_Remove_ALL excludes OO, RI, and low-occurrence questions.
    # These methods are more sensitive to sparse columns than HGB.
    RI_set_ALL           = set(RI_Remove_ALL)
    predictors_R_T2B_ALL = [c for c in T2B_cols      if c not in RI_set_ALL]
    predictors_R_TEN_ALL = [c for c in all_10pt_cols  if c not in RI_set_ALL]
    predictors_ALL       = predictors_R_T2B_ALL + predictors_R_TEN_ALL

    lr_df      = compute_linear_weights_LR_v2(
        df, predictors_ALL, target_column, missing_threshold)
    johnson_df = compute_johnson_weights_v2(
        df, predictors_ALL, target_column, missing_threshold)
    lasso_df   = compute_lasso_importance_v2(
        df, predictors_ALL, target_column, missing_threshold)
    rf_df      = compute_random_forest_importance_v2(
        df, predictors_ALL, target_column, missing_threshold)

    # Merge all methods into a single comparison table
    merged_df = lr_df.merge(johnson_df, on='Variable', how='outer')
    merged_df = merged_df.merge(rf_df,     on='Variable', how='outer')
    merged_df = merged_df.merge(lasso_df,  on='Variable', how='outer')
    # Uncomment to include signed OLS coefficients:
    # ols_df    = ols_signed_coefficients(df, predictors_ALL, target_column)
    # merged_df = merged_df.merge(ols_df, on='Variable', how='outer')

    DA_file_path = f"{path}_DA_Combined_{save_name}.xlsx"
    merged_df.to_excel(DA_file_path, index=False)
    print(f"  Driver analysis saved to {DA_file_path}")

# ── CATA Pairwise Significance (across all products) ─────────────────────────
all_cata_results = batch_compare_cata_mcnemar(
    df_mapped_list_of_df, df_mapped_list, CATA_cols, path)

print("\n✅ Pipeline complete.")